[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/upm-classes/ai_medical_imaging_2026/blob/main/notebooks/student/NB00_environment.ipynb)

# NB00 · Environment and reproducibility
**Artificial Intelligence in Medical Imaging** · Day 1 · run in class (about 10 min)

| | |
|---|---|
| Runtime | CPU is enough (GPU optional) |
| Data | none |
| You will | check the environment, write a `set_seed` function, create a project folder, save an environment report |
| Checkpoint | the last cell prints **Environment OK** |

**Why this matters.** A result that nobody can reproduce is not a result. In medical imaging, a model that changes between two runs cannot be audited. Today you build the three habits we use all week: print versions, fix seeds, keep a tidy folder.

## 1. Versions and hardware

In [13]:
import sys, platform, random, os, json
from pathlib import Path
import numpy as np
# TODO 1: import torch
import torch

print("Python  :", sys.version.split()[0])
print("PyTorch :", torch.__version__)
print("NumPy   :", np.__version__)
print("Platform:", platform.platform())

Python  : 3.13.18
PyTorch : 2.14.1+cpu
NumPy   : 2.4.6
Platform: Linux-6.6.159+-x86_64-with-glibc2.39


In [14]:
# TODO 2: is a GPU available? (hint: torch.cuda.is_available())
is_cuda = None
device = torch.device("cuda" if is_cuda else "cpu")
print("GPU available:", is_cuda, "| device:", device)
if is_cuda:
    print("GPU name:", torch.cuda.get_device_name(0))

GPU available: None | device: cpu


> **No GPU? Nothing changes.** All labs of the week run on CPU. A GPU only makes some of them faster (Runtime → Change runtime type).

## 2. Seeds

Training starts from random weights, shuffles the data and may use random augmentation. A **seed** makes all of that repeatable. We need to seed *every* library that draws random numbers.

In [22]:
def set_seed(seed: int = 42) -> None:
    """Seed Python, NumPy and PyTorch (CPU and GPU)."""
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    # TODO 3: seed NumPy (hint: np.random.seed)
    np.random.seed(seed)
    # TODO 4: seed PyTorch on the CPU (hint: torch.manual_seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

set_seed(42)
print("seed set")

seed set


### Checkpoint 1 · same seed, same numbers
Predict first: what will the two NumPy draws below be, equal or different? Then run.

In [16]:
set_seed(0); a = np.random.rand(3); ta = torch.rand(3)
set_seed(0); b = np.random.rand(3); tb = torch.rand(3)
set_seed(1); c = np.random.rand(3)

assert np.allclose(a, b), "NumPy: same seed must give the same numbers. Did you seed NumPy?"
assert torch.equal(ta, tb), "PyTorch: same seed must give the same numbers. Did you seed torch?"
assert not np.allclose(a, c), "A different seed should give different numbers"
print("Checkpoint 1 passed:", a.round(3))

Checkpoint 1 passed: [0.549 0.715 0.603]


> **Reflection 1.** Seeds make a run *repeatable*, not *correct*. If two seeds give accuracies 0.84 and 0.87, which one do you report? (Answer in your pair: we come back to this in the Day-2 tuning race.)

> **Plus.** On a GPU some operations stay non-deterministic even with these flags. Search the PyTorch documentation for `torch.use_deterministic_algorithms` and note one trade-off.

## 3. A tidy project folder

In [36]:
project_dirs = ["data", "outputs", "checkpoints"]
for d in project_dirs:
    # TODO 5: create the folder if it does not exist (hint: Path(d).mkdir(exist_ok=True))

    dir_path = Path(d)
    dir_path.mkdir(exist_ok=True)
    print("ready:", d)


ready: data
ready: outputs
ready: checkpoints


## 4. An environment report (copy this habit into your project)
Save the versions next to your results. Six months later, it tells you what produced them.

In [37]:
report = {
    "python": sys.version.split()[0],
    "torch": torch.__version__,
    "numpy": np.__version__,
    "cuda": bool(is_cuda),
    "seed": 42,
}
Path("outputs/env_report.json").write_text(json.dumps(report, indent=2))
print(json.dumps(report, indent=2))

{
  "python": "3.13.18",
  "torch": "2.14.1+cpu",
  "numpy": "2.4.6",
  "cuda": false,
  "seed": 42
}


## 5. Final check

In [40]:
is_cuda = torch.cuda.is_available()
assert isinstance(is_cuda, bool)
assert all(Path(d).is_dir() for d in project_dirs)
assert Path("outputs/env_report.json").exists()
print("Environment OK")
print(f"Python {sys.version.split()[0]} | PyTorch {torch.__version__} | NumPy {np.__version__}")

Environment OK
Python 3.13.18 | PyTorch 2.14.1+cpu | NumPy 2.4.6


**Done.** Put your cup on green and help a neighbour.

*Next:* NB01 (DICOM, Hounsfield units and windowing).

### If something fails
| Symptom | Fix |
|---|---|
| `ModuleNotFoundError: torch` | You are not in Colab. Use Colab, or install PyTorch following pytorch.org |
| `Environment OK` never prints | An earlier TODO is still `None` or `...`: read the assertion message |
| Runtime disconnected | Runtime → Reconnect, then Run all |